In [1]:
import pandas as pd
import numpy as np

# توليد بيانات ملوثة وعشوائية للاختبار
np.random.seed(10)
n_rows = 1000

data = {
    'Transaction_ID': [f'TXN_{1000 + i}' for i in range(n_rows)],
    'Date': np.random.choice(['2026/05/01', '02-05-2026', '2026-05-03', None, '2026/05/05'], size=n_rows),
    'Customer_Detail': np.random.choice(['C101 - Ahmed', 'C102-Mohamed', 'C103 -  Ali ', 'UNKNOWN', None], size=n_rows),
    'Amount': np.random.choice([1500, -250, 3000, '5000 USD', 0, None, -120], size=n_rows),
    'Status': np.random.choice(['Completed', 'completed', ' Pending', 'Failed', 'FAILED'], size=n_rows)
}

df = pd.DataFrame(data)

# عمل صفوف مكررة بالقصـد (تكرار 20 صف)
df = pd.concat([df, df.iloc[10:30]], ignore_index=True)

# حفظ الملف
df.to_csv('dirty_financial_data.csv', index=False)
print("⚠️ تم إنشاء ملف البيانات المليء بالأخطاء بنجاح باسم: dirty_financial_data.csv")
print("جاهزون لبدء رحلة التنظيف!")

⚠️ تم إنشاء ملف البيانات المليء بالأخطاء بنجاح باسم: dirty_financial_data.csv
جاهزون لبدء رحلة التنظيف!


In [2]:
import pandas as pd

# 1. قراءة الملف الملوث
df = pd.read_csv('dirty_financial_data.csv')

print("📊 حجم البيانات قبل التنظيف:", df.shape)

# 2. إزالة الصفوف المكررة
df = df.drop_duplicates()

# 3. توحيد صيغ التواريخ (Pandas هيتعرف على التواريخ المختلفة ويوحدها)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# 4. تنظيف عمود المبالغ (Amount)
# تحويل القيم لنصوص، مسح كلمة USD، ثم تحويلها لأرقام
df['Amount'] = df['Amount'].astype(str).str.replace(' USD', '')
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

# 5. فصل كود العميل عن اسمه (بناءً على علامة - )
df[['Customer_Code', 'Customer_Name']] = df['Customer_Detail'].str.split('-', n=1, expand=True)

# تنظيف المسافات الزائدة من الأكواد والأسماء
df['Customer_Code'] = df['Customer_Code'].str.strip()
df['Customer_Name'] = df['Customer_Name'].str.strip()

# حذف العمود القديم المدمج لأننا مش محتاجينه خلاص
df = df.drop('Customer_Detail', axis=1)

# 6. توحيد عمود الحالة (Status)
# مسح المسافات الزائدة وجعل أول حرف فقط كابيتال
df['Status'] = df['Status'].str.strip().str.title()

# 7. التعامل مع القيم المفقودة (Missing Values)
df['Customer_Code'] = df['Customer_Code'].fillna('Unknown')
df['Customer_Name'] = df['Customer_Name'].fillna('Unknown')
df['Amount'] = df['Amount'].fillna(0)

# 8. حفظ الملف النظيف النهائي
df.to_csv('cleaned_financial_data.csv', index=False)

print("✅ حجم البيانات بعد إزالة التكرار:", df.shape)
print("\n✨ أول 5 صفوف بعد التنظيف السحري:")
print(df.head())

📊 حجم البيانات قبل التنظيف: (1020, 5)
✅ حجم البيانات بعد إزالة التكرار: (1000, 6)

✨ أول 5 صفوف بعد التنظيف السحري:
  Transaction_ID       Date  Amount     Status Customer_Code Customer_Name
0       TXN_1000 2026-02-05  -250.0     Failed          C102       Mohamed
1       TXN_1001        NaT  3000.0     Failed       UNKNOWN       Unknown
2       TXN_1002        NaT  -120.0     Failed          C102       Mohamed
3       TXN_1003 2026-02-05  -250.0     Failed          C103           Ali
4       TXN_1004        NaT  1500.0  Completed          C103           Ali
